In [1]:
from pathlib import Path
import polars as pl

DATA_DIR = Path("/Users/matiaslee/LOCAL_FILES_SAFE/Dr Facelli Project 2025/AS_DS")
EXPORT_DIR = DATA_DIR / "AS_DS/exports_practicum1"

AS_PATH = DATA_DIR / "as_per_patient_20260216.parquet"
COVID_PATH = DATA_DIR / "covid_infection_summary_20260216.parquet"
AS_COVID_PATH = EXPORT_DIR / "as_covid_dates_df.parquet"
HLA_PATH = EXPORT_DIR / "hla_summary_df.parquet"
ALL_AS_EVENTS_PATH = EXPORT_DIR / "all_as_events.parquet"




In [2]:
as_covid_df = pl.read_parquet(AS_COVID_PATH)
hla_df = pl.read_parquet(HLA_PATH)

summary_panel = (
    as_covid_df
    .join(
        hla_df.select("patient_id", "has_HLA_AS", "has_AS", "has_AS_dx", "first_HLA_AS_date"),
        on="patient_id",
        how="left",
    )
    .with_columns([
        pl.col("has_HLA_AS").fill_null(False),
        pl.col("has_AS").fill_null(True),   # everyone in as_covid_df should already be AS
        pl.col("has_AS_dx").fill_null(False),
        pl.lit(True).alias("has_COVID"),
    ])
)

summary_panel.head()

patient_id,first_AS_date,last_AS_date,first_covid_date,last_covid_date,has_HLA_AS,has_AS,has_AS_dx,first_HLA_AS_date,has_COVID
str,date,date,date,date,bool,bool,bool,date,bool
"""dB_C""",2019-10-20,2025-05-13,2021-01-08,2021-01-08,false,true,true,null,true
"""eQfE""",2020-01-08,2020-01-08,2022-06-21,2022-06-21,false,true,true,null,true
"""HQ5""",2019-04-20,2020-02-21,2020-12-02,2020-12-02,false,true,true,null,true
"""9gd""",2015-11-27,2024-08-16,2020-12-31,2024-01-25,false,true,true,null,true
"""3wHf""",2024-01-09,2024-01-09,2024-02-04,2024-02-04,false,true,true,null,true


In [3]:
n_as_covid = summary_panel["patient_id"].n_unique()
n_hla_pos = summary_panel.filter(pl.col("has_HLA_AS")).height

print("AS + COVID patients:", n_as_covid)
print("HLA-B27+ within AS + COVID:", n_hla_pos)

AS + COVID patients: 18273
HLA-B27+ within AS + COVID: 4054


In [4]:
def cohort(label, condition):
    n = summary_panel.filter(condition).height
    return {
        "cohort": label,
        "n_patients": int(n),
        "pct_of_AS_COVID": round(100 * n / n_as_covid, 2) if n_as_covid else None,
    }

In [5]:
rows = []

rows.append(
    cohort(
        "All AS + COVID",
        pl.lit(True)
    )
)

rows.append(
    cohort(
        "HLA-B27 positive",
        pl.col("has_HLA_AS")
    )
)

rows.append(
    cohort(
        "AS BEFORE first COVID infection",
        pl.col("first_AS_date").is_not_null()
        & (pl.col("first_AS_date") < pl.col("first_covid_date"))
    )
)

rows.append(
    cohort(
        "AS ONLY AFTER first COVID infection",
        pl.col("first_AS_date").is_not_null()
        & (pl.col("first_AS_date") > pl.col("first_covid_date"))
    )
)

rows.append(
    cohort(
        "AS ON same day as first COVID infection",
        pl.col("first_AS_date").is_not_null()
        & (pl.col("first_AS_date") == pl.col("first_covid_date"))
    )
)

rows.append(
    cohort(
        "HLA-B27+ BEFORE first COVID infection",
        pl.col("has_HLA_AS")
        & pl.col("first_HLA_AS_date").is_not_null()
        & (pl.col("first_HLA_AS_date") < pl.col("first_covid_date"))
    )
)

rows.append(
    cohort(
        "HLA-B27+ ONLY AFTER first COVID infection",
        pl.col("has_HLA_AS")
        & pl.col("first_HLA_AS_date").is_not_null()
        & (pl.col("first_HLA_AS_date") > pl.col("first_covid_date"))
    )
)

rows.append(
    cohort(
        "HLA-B27+ ON same day as first COVID infection",
        pl.col("has_HLA_AS")
        & pl.col("first_HLA_AS_date").is_not_null()
        & (pl.col("first_HLA_AS_date") == pl.col("first_covid_date"))
    )
)

summary_df = pl.DataFrame(rows)
summary_df

cohort,n_patients,pct_of_AS_COVID
str,i64,f64
"""All AS + COVID""",18273,100.0
"""HLA-B27 positive""",4054,22.19
"""AS BEFORE first COVID infectio…",11778,64.46
"""AS ONLY AFTER first COVID infe…",6100,33.38
"""AS ON same day as first COVID …",395,2.16
"""HLA-B27+ BEFORE first COVID in…",2705,14.8
"""HLA-B27+ ONLY AFTER first COVI…",1340,7.33
"""HLA-B27+ ON same day as first …",9,0.05


**Did COVID increase AS diagnosis frequency in people with AS?**
The most logical primary analysis is on patients who already had AS before COVID.

In [6]:
from pathlib import Path
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path("/Users/matiaslee/LOCAL_FILES_SAFE/Dr Facelli Project 2025/AS_DS")
EXPORT_DIR = DATA_DIR / "AS_DS/exports_practicum1"

AS_COVID_PATH = EXPORT_DIR / "as_covid_dates_df.parquet"
ALL_AS_EVENTS_PATH = EXPORT_DIR / "all_as_events.parquet"

WINDOW_DAYS = 365 * 0.5

## Primary Analysis

Question: Did COVID increase AS diagnosis frequency in people with AS?

Primary cohort:
- patients with AS evidence before first COVID infection

Outcome:
- number of AS-flagging events in the symmetric window before and after first COVID

Current window:
- 2 years before vs 2 years after first COVID
- same-day events excluded

In [7]:
as_covid_df = pl.read_parquet(AS_COVID_PATH)
all_as_events_df = pl.read_parquet(ALL_AS_EVENTS_PATH)

print("as_covid_df:", as_covid_df.shape)
print("all_as_events_df:", all_as_events_df.shape)

as_covid_df.head(), all_as_events_df.head()

as_covid_df: (18273, 5)
all_as_events_df: (711220, 2)


(shape: (5, 5)
 ┌────────────┬───────────────┬──────────────┬──────────────────┬─────────────────┐
 │ patient_id ┆ first_AS_date ┆ last_AS_date ┆ first_covid_date ┆ last_covid_date │
 │ ---        ┆ ---           ┆ ---          ┆ ---              ┆ ---             │
 │ str        ┆ date          ┆ date         ┆ date             ┆ date            │
 ╞════════════╪═══════════════╪══════════════╪══════════════════╪═════════════════╡
 │ dB_C       ┆ 2019-10-20    ┆ 2025-05-13   ┆ 2021-01-08       ┆ 2021-01-08      │
 │ eQfE       ┆ 2020-01-08    ┆ 2020-01-08   ┆ 2022-06-21       ┆ 2022-06-21      │
 │ HQ5        ┆ 2019-04-20    ┆ 2020-02-21   ┆ 2020-12-02       ┆ 2020-12-02      │
 │ 9gd        ┆ 2015-11-27    ┆ 2024-08-16   ┆ 2020-12-31       ┆ 2024-01-25      │
 │ 3wHf       ┆ 2024-01-09    ┆ 2024-01-09   ┆ 2024-02-04       ┆ 2024-02-04      │
 └────────────┴───────────────┴──────────────┴──────────────────┴─────────────────┘,
 shape: (5, 2)
 ┌────────────┬────────────┐
 │ patient_id ┆ 

In [8]:
all_as_events_df = all_as_events_df.unique(subset=["patient_id", "as_date"])

all_as_events_df.select([
    pl.len().alias("n_rows"),
    pl.col("patient_id").n_unique().alias("n_unique_patients"),
    pl.col("as_date").min().alias("min_as_date"),
    pl.col("as_date").max().alias("max_as_date"),
])

n_rows,n_unique_patients,min_as_date,max_as_date
u32,u32,date,date
708714,114246,1950-01-01,2025-05-24


In [9]:
pre_existing_as_covid = (
    as_covid_df
    .filter(pl.col("first_AS_date") < pl.col("first_covid_date"))
)

pre_existing_as_covid.select([
    pl.len().alias("n_patients_pre_existing_AS_before_COVID")
])

n_patients_pre_existing_AS_before_COVID
u32
11778


In [10]:
as_events_primary = (
    all_as_events_df
    .join(
        pre_existing_as_covid.select(["patient_id", "first_covid_date"]),
        on="patient_id",
        how="inner",
    )
    .with_columns(
        (
            pl.col("as_date").cast(pl.Int64) - pl.col("first_covid_date").cast(pl.Int64)
        ).alias("delta_days_from_covid")
    )
)

as_events_primary.head()

patient_id,as_date,first_covid_date,delta_days_from_covid
str,date,date,i64
"""jQtc""",2018-06-04,2020-02-25,-631
"""SQ5a""",2024-10-29,2023-03-30,579
"""YRiJ""",2022-08-05,2022-08-23,-18
"""PxUXB""",2018-12-11,2024-05-23,-1990
"""Mgpb""",2021-06-16,2021-08-14,-59


In [11]:
as_events_windowed = (
    as_events_primary
    .with_columns(
        pl.when(
            (pl.col("delta_days_from_covid") >= -WINDOW_DAYS) &
            (pl.col("delta_days_from_covid") <= -1)
        )
        .then(pl.lit("before"))
        .when(
            (pl.col("delta_days_from_covid") >= 1) &
            (pl.col("delta_days_from_covid") <= WINDOW_DAYS)
        )
        .then(pl.lit("after"))
        .otherwise(None)
        .alias("period")
    )
    .filter(pl.col("period").is_not_null())
)

as_events_windowed.select([
    pl.len().alias("n_rows_in_window"),
    pl.col("delta_days_from_covid").min().alias("min_delta"),
    pl.col("delta_days_from_covid").max().alias("max_delta"),
])

n_rows_in_window,min_delta,max_delta
u32,i64,i64
20924,-182,182


In [12]:
per_patient_counts = (
    as_events_windowed
    .group_by("patient_id")
    .agg([
        (pl.col("period") == "before").sum().alias("n_AS_evidence_before"),
        (pl.col("period") == "after").sum().alias("n_AS_evidence_after"),
    ])
)

per_patient_counts = (
    pre_existing_as_covid
    .select("patient_id")
    .join(per_patient_counts, on="patient_id", how="left")
    .with_columns([
        pl.col("n_AS_evidence_before").fill_null(0).cast(pl.Int64),
        pl.col("n_AS_evidence_after").fill_null(0).cast(pl.Int64),
    ])
)

per_patient_counts.head()

patient_id,n_AS_evidence_before,n_AS_evidence_after
str,i64,i64
"""dB_C""",1,0
"""eQfE""",0,0
"""HQ5""",0,0
"""9gd""",2,2
"""3wHf""",1,0


In [13]:
primary_summary = per_patient_counts.select([
    pl.len().alias("n_patients"),
    pl.col("n_AS_evidence_before").sum().alias("total_AS_evidence_before"),
    pl.col("n_AS_evidence_after").sum().alias("total_AS_evidence_after"),
    pl.col("n_AS_evidence_before").mean().alias("mean_AS_evidence_before"),
    pl.col("n_AS_evidence_after").mean().alias("mean_AS_evidence_after"),
    pl.col("n_AS_evidence_before").median().alias("median_AS_evidence_before"),
    pl.col("n_AS_evidence_after").median().alias("median_AS_evidence_after"),
    (pl.col("n_AS_evidence_after") > pl.col("n_AS_evidence_before")).sum().alias("n_after_gt_before"),
    (pl.col("n_AS_evidence_after") == pl.col("n_AS_evidence_before")).sum().alias("n_after_eq_before"),
    (pl.col("n_AS_evidence_after") < pl.col("n_AS_evidence_before")).sum().alias("n_after_lt_before"),
])

primary_summary

n_patients,total_AS_evidence_before,total_AS_evidence_after,mean_AS_evidence_before,mean_AS_evidence_after,median_AS_evidence_before,median_AS_evidence_after,n_after_gt_before,n_after_eq_before,n_after_lt_before
u32,i64,i64,f64,f64,f64,f64,u32,u32,u32
11778,10931,9993,0.928086,0.848446,0.0,0.0,1645,7442,2691


In [14]:
per_patient_counts = per_patient_counts.with_columns(
    (pl.col("n_AS_evidence_after") - pl.col("n_AS_evidence_before")).alias("delta_AS_evidence")
)

delta_summary = per_patient_counts.select([
    pl.col("delta_AS_evidence").mean().alias("mean_delta"),
    pl.col("delta_AS_evidence").median().alias("median_delta"),
    (pl.col("delta_AS_evidence") > 0).sum().alias("n_increased"),
    (pl.col("delta_AS_evidence") == 0).sum().alias("n_unchanged"),
    (pl.col("delta_AS_evidence") < 0).sum().alias("n_decreased"),
])

delta_summary

mean_delta,median_delta,n_increased,n_unchanged,n_decreased
f64,f64,u32,u32,u32
-0.07964,0.0,1645,7442,2691


In [15]:
n_total = per_patient_counts.height

direction_pct = per_patient_counts.select([
    ((pl.col("delta_AS_evidence") > 0).sum() / n_total * 100).alias("pct_increased"),
    ((pl.col("delta_AS_evidence") == 0).sum() / n_total * 100).alias("pct_unchanged"),
    ((pl.col("delta_AS_evidence") < 0).sum() / n_total * 100).alias("pct_decreased"),
])

direction_pct

pct_increased,pct_unchanged,pct_decreased
f64,f64,f64
13.966718,63.1856,22.847682


In [16]:
per_patient_counts.to_pandas()[["n_AS_evidence_before", "n_AS_evidence_after"]].describe()

,n_AS_evidence_before,n_AS_evidence_after
count,11778.000000,11778.000000
mean,0.928086,0.848446
std,2.246998,2.340079
min,0.000000,0.000000
25%,0.000000,0.000000
50%,0.000000,0.000000
75%,1.000000,1.000000
max,81.000000,38.000000


In [17]:
# ----------------------
# Timing split within a 4-year window
# ----------------------

WINDOW_DAYS_4Y = 365 * 4

as_covid_4y = (
    as_covid_df
    .with_columns(
        (
            pl.col("first_AS_date").cast(pl.Int64)
            - pl.col("first_covid_date").cast(pl.Int64)
        ).alias("delta_days")
    )
    .filter(pl.col("delta_days").abs() <= WINDOW_DAYS_4Y)
)

timing_4y_summary = as_covid_4y.select([
    pl.len().alias("n_patients_within_4y"),
    (pl.col("delta_days") < 0).sum().alias("n_AS_before_COVID"),
    (pl.col("delta_days") > 0).sum().alias("n_AS_after_COVID"),
    (pl.col("delta_days") == 0).sum().alias("n_same_day"),
])

timing_4y_summary

n_patients_within_4y,n_AS_before_COVID,n_AS_after_COVID,n_same_day
u32,u32,u32,u32
13209,7194,5620,395


In [18]:
n_total_4y = as_covid_4y.height

timing_4y_table = pl.DataFrame([
    {
        "Group": "AS before COVID (within 4 years)",
        "N": as_covid_4y.filter(pl.col("delta_days") < 0).height,
        "Pct": round(100 * as_covid_4y.filter(pl.col("delta_days") < 0).height / n_total_4y, 2),
    },
    {
        "Group": "AS after COVID (within 4 years)",
        "N": as_covid_4y.filter(pl.col("delta_days") > 0).height,
        "Pct": round(100 * as_covid_4y.filter(pl.col("delta_days") > 0).height / n_total_4y, 2),
    },
    {
        "Group": "Same day (within 4 years)",
        "N": as_covid_4y.filter(pl.col("delta_days") == 0).height,
        "Pct": round(100 * as_covid_4y.filter(pl.col("delta_days") == 0).height / n_total_4y, 2),
    },
])

timing_4y_table

Group,N,Pct
str,i64,f64
"""AS before COVID (within 4 year…",7194,54.46
"""AS after COVID (within 4 years…",5620,42.55
"""Same day (within 4 years)""",395,2.99
